<a href="https://colab.research.google.com/github/Jacelmoffett1/Website-Tool-Checker/blob/main/MP_9.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>


| Platform | Category | Signals | Source |
| :---------------- | :------: | :---- | :-----:|
| Drupal | CMS  |Generator = drupal, data-drupal,| https://www.ncaa.com/ |
| Wix | Website/Page Builder |generator=wix,wixdesktopviewport, css to wix|https://www.ilovechickpea.ca/|
| Weebly | Website/Page Builder | look for weebly|https://womenof1000ad.weebly.com/|
| Shopify | E-Commerce |in preconnect it says shopify and 231 other places. |https://pattersonfan.com/|
| Joomla | CMS  |Look for joomla in link to stylesheets or gen, or scripts |https://pawnrealm.com/|
| Webflow | Website/Page Builder |Look for webflow in style sheet links |https://www.bunkrdesign.co.uk/|
| Ghost | CMS  | Look for ghost in meta and scripts from storage.ghost |https://www.readtangle.com/|
| HubSpot CMS | CMS | Look for hubspot in scripts and gen |https://www.blendb2b.com/work/robin-radar-website|
| Adobe Experience Manager | CMS | look for a stylesheet to adobe-cms and also in various link and script tags|https://www.ibm.com/us-en|
| WordPress | CMS  | look fro wordpress in gen and look for wp in links, scripts and styles|https://www.taylorswift.com/|
| Squarespace | Website/Page Builder | Look for squarespace or sqsp in style sheets links etc |https://www.fatchoynyc.com/|
| ShowIt | Website/Page Builder	 |Look for showit in style sheet links |https://jasminestar.com/|
| Framer | Website/Page Builder	 | There usually a comment at the top that labels it as built in framer, gen tag, framer in stylesheet, framer-text is a class|https://kynejang.com/|
| Prismic | CMS |look for prismic in scripts |https://mcdonalds.es/|
| ContentStack | CMS | Look for blt before long number sequences in meta|https://www.hawaiianairlines.com/|
| Contentful | CMS | look for ctf in from on image src images.ctf....|https://www.kfc.com/|
| Divi | Website/Page Builder |divi in style sheet link or in id|https://kerlinbbq.com/|
| site123 | Website/Page Builder	 |look for the s123 in class names in meta, links|https://www.federicosfood.com/|
| Strikingly | Website/Page Builder	 |Look for strikingly in link tags, (stylesheet link) |https://www.cpclondon.com/|
| godaddy | Website/Page Builder |look for go daddy in gen or godaddy in p tags labeled filler@godaddy.com|https://cocoriverorganics.com/|

In [ ]:
import requests
from bs4 import BeautifulSoup
import re
# from urllib.parse import urlparse

In [ ]:
!pip install google-colab-selenium -q

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 11.8/11.8 MB 56.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 511.8/511.8 kB 23.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 135.7/135.7 kB 8.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4.9/4.9 MB 62.0 MB/s eta 0:00:00


In [ ]:
import google_colab_selenium as gs

In [ ]:
def analyze_website_selenium():
    while True:
      url = input("Enter a Url ").strip()
      if not(url.startswith("https://") or url.startswith("http://")):
          url = "https://" + url
      if url != "":
          break
      print("Please enter a correct Url")
    driver = None
    try:
        driver = gs.Chrome()
        driver.get(url)

        driver.implicitly_wait(5)
        full_html_content = driver.page_source

        # 1. Platform Detection
        detect_platform(full_html_content)

        # 2. Google Tag Detection
        detect_google_tags(full_html_content)

        # 3. Meta Pixel Detection
        detect_meta(full_html_content)

    except Exception as e:
        print(f"An error occurred with Selenium: {str(e)}")
    finally:
        if driver:
            driver.quit()


In [ ]:
def analyze_website():
    while True:
      url = input("Enter a Url ").strip()
      if not(url.startswith("https://") or url.startswith("http://")):
          url = "https://" + url
      if url != "":
          break
      print("Please enter a correct Url")

    custom_headers = {
    "User-Agent": "Mozilla/5.0 (Windows NT 11.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/134.0.6998.166 Safari/537.36",
    "Authorization": "Bearer YOUR_API_TOKEN",
    "Accept-Language": "en-US,en;q=0.9",
    "Accept": "text/html,application/xhtml+xml,application/xml;q=0.9,image/avif,image/webp,image/apng,*/*;q=0.8"
    }


    try:
        response = requests.get(url, timeout=15, allow_redirects=True,headers=custom_headers)
        response.raise_for_status()
        soup = BeautifulSoup(response.text, "html.parser")
        text = soup.get_text()

        detect_platform(response.text)


        detect_google_tags(response.text)

        detect_meta(response.text)

    except requests.exceptions.SSLError:
        print("Error: SSL certificate verification failed. The site may have security issues.")
    except requests.exceptions.ConnectionError:
        print("Error: DNS or Connection failure. Check if the domain is valid.")
    except requests.exceptions.HTTPError as e:
        print(f"HTTP Error encountered: {e.response.status_code} {e.response.reason}")
    except Exception as e:
        print(f"An error occurred: {str(e)}")


In [ ]:
def detect_platform(full_html_content):
    full_html_content = full_html_content.lower()
    platform_rules = {
        'WordPress': [r'wp-content', r'wp-includes', r'generator" content="WordPress'],
        'Shopify': [r'shopify\.com', r'cdn\.shopify\.com', r'shopify-payment-button'],
        'Squarespace': [r'static1\.squarespace\.com', r'squarespace-announcement-bar', r'sqsp'],
        'Wix': [r'wix\.com', r'_wix_', r'wix-style-list'],
        'Drupal': [r'Drupal', r'data-drupal-', r'sites/default/files'],
        'Joomla': [r'joomla', r'index\.php\?option=com_',r'joomla-script',r'joomla-alert'],
        'Webflow': [r'webflow\.com', r'w-css-',r'data-wf-domain',r'data-wf-site',r'data-wf-page'],
        'HubSpot': [r'hubspot\.com', r'hs-scriptloader',r'hs-content',r'hs-site',r'hubspot'],
        'Adobe Experience Manager': [r'etc\.clientlibs', r'adobe-cms'],
        'Ghost': [r'ghost\.org', r'ghost-sdk',r'storage.ghost'],
        'Showit': [r'showit\.com', r'showit_'],
        'Framer': [r'framer\.com', r'framer-text', r'built in framer',r'framer'],
        'Prismic': [r'prismic\.io', r'static\.cdn\.prismic\.io',r'prismic'],
        'Contentstack': [r'blt[a-f0-9]{16}'],
        'Contentful': [r'images\.ctfassets', r'contentful\.com'],
        'Divi': [r'et-divi', r'divi-style'],
        'Site123': [r's123-js', r'site123\.com'],
        'Strikingly': [r'strikingly\.com', r's-common-css',r'strikingly'],
        'GoDaddy': [r'godaddy', r'filler@godaddy\.com'],
        'Weebly': [r'weebly\.com', r'weebly-theme',r'weebly-footer',r'wsite']
    }

    matches = []
    for name, patterns in platform_rules.items():
        hits = [p for p in patterns if re.search(p, full_html_content)]
        if hits:
            matches.append({'name': name, 'hits': hits})
    if not matches:
        print("Detected Platform: Unknown")
        print("Evidence: No standard CMS or builder signals found.")
    else:
        primary = matches[0]
        if len(primary['hits']) > 2:
            confidence = "High"
        elif len(primary['hits']) > 1:
            confidence = "Medium"
        else:
            confidence = "Low"
        print(f"Detected Platform: {primary['name']}")
        print(f"Evidence: Found matches for {', '.join(primary['hits'][:3])}")
        print(f"Confidence Level: {confidence}")
        if len(matches) > 1:
            print(f"Secondary possibilities: {', '.join([m['name'] for m in matches[1:]])}")



In [ ]:
def detect_google_tags(html):
    ga4_ids = 'id=G-'in html
    ua_ids = 'id=UA-' in html
    gt_ids = 'id=GT-'in html
    gtm_ids = 'id=GTM-' in html
    all_ga = (ga4_ids or ua_ids or gt_ids or gtm_ids)
    if all_ga:
      print(f"Google Tags Detected: Yes")
      get_GoogleID(html)
    else:
      print(f"Google Tags Detected: No")
      print(f"No Google Tag signals found.")

In [ ]:
def get_GoogleID(html):
    google_pattern = r'(G-[A-Z0-9]{10}|UA-\d{4,9}-\d{1,4}|GT-[A-Z0-9]+|GTM-[A-Z0-9]+)'
    google_ids = re.findall(google_pattern, html)
    if google_ids:
      unique_google_ids = sorted(list(set(google_ids)))
      print(f"Google ID's: {unique_google_ids}" )

In [ ]:
def detect_meta(html):
    html = html.lower()
    meta_rules = [
        r'fbevents\.js',
        r'fbq\(',
        r"fbq\('init',"
    ]

    detected_rules = []
    for rule in meta_rules:
        if re.search(rule, html):
            detected_rules.append(rule)

    if detected_rules:
        print(f"Meta Pixel Detected: Yes")
        get_metaID(html)
        print(f"Evidence: Found matches for {', '.join(detected_rules)}")
        if len(detected_rules) > 1:
            print("Confidence Level: High")
        else:
          print("Confidence Level: Medium")
    else:
        print("Meta Pixel Detected: No")
        print("No Meta Pixel signals found.")


In [ ]:
def get_metaID(html):
    pixel_pattern = r"(?:fbq\('init',\s*'|id=)(\d{15,16})"
    pixel_ids = re.findall(pixel_pattern, html)
    if pixel_ids:
      print(f"Meta ID's: {pixel_ids}" )


In [ ]:
# analyze_website()
analyze_website_selenium()

Enter a Url https://countryfriedcreative.com/


<IPython.core.display.Javascript object>

Detected Platform: WordPress
Evidence: Found matches for wp-content, wp-includes
Confidence Level: Medium
Secondary possibilities: Divi
Google Tags Detected: Yes
Google ID's: ['G-RNZYT9JD3M', 'GTM-TFXX8LKS']
Meta Pixel Detected: Yes
Meta ID's: ['3370956352999327', '3370956352999327', '6619506900210929']
Evidence: Found matches for fbevents\.js, fbq\(, fbq\('init',
Confidence Level: High


1. A lot of websites will hide/remove their generator tag for security reasons, as the less someone trying to lets say hack their website the better.
2. I'd say evidence that was longer in length as it would be harder to find a match for a string with a long length rather than a short length string.
3. Despite using selenium with a delay in the hopes of capturing a websites full html after all javascript has been executed and the full page is loaded, some websites still got around this and sent me incomplete or a fake html page.
4. I prioritized number of pieces of evidence so that the more unique piecies of evidence there is for a specific tool the higher confidence I have that it was used.
5. I would try to find a way to get around all errors, such as certain websites giving me http 403 errors, and I'd also try to add more pieces of evidence for each tool.



#Example Outputs:
1. Enter a Url https://www.luckycatcollective.com/ \
Initialized Chromedriver\
Detected Platform: Shopify\
Evidence: Found matches for shopify\.com, cdn\.shopify\.com, shopify-payment-button\
Confidence Level: High\
Google Tag Manager not Detected \
Meta Pixel Detected: Yes\
Evidence: Found matches for fbevents\.js\
Confidence Level: Medium
2. Enter a Url https://countryfriedcreative.com/ \
Initialized Chromedriver\
Detected Platform: WordPress\
Evidence: Found matches for wp-content, wp-includes\
Confidence Level: Medium\
Secondary possibilities: Divi\
Google Ids: ga4  \
Meta Pixel Detected: No\
No Meta Pixel signals found.
3. This is an example where we get blockedby the host to get their real html. The website is built on prismic.\
Enter a Url https://mcdonalds.es/ \
Initialized Chromedriver\
Detected Platform: Unknown\
Evidence: No standard CMS or builder signals found.\
Google Tag Manager not Detected \
Meta Pixel Detected: No\
No Meta Pixel signals found.
4. Enter a Url https://www.allkeyshop.com/blog/en-us/\
Initialized Chromedriver\
Detected Platform: WordPress\
Evidence: Found matches for wp-content, wp-includes\
Confidence Level: Medium\
Google Ids: ga4  \
Meta Pixel Detected: No\
No Meta Pixel signals found.
5. Enter a Url https://jasminestar.com/\
Initialized Chromedriver\
Detected Platform: Showit\
Evidence: Found matches for showit\.com\
Confidence Level: Low\
Google Ids: ga4  \
Meta Pixel Detected: Yes\
Meta ID's: ['1047994071927834', '1014777945617059']\
Evidence: Found matches for fbevents\.js, fbq\(\
Confidence Level: High
6. Enter a Url https://www.federicosfood.com/ \
Initialized Chromedriver\
Detected Platform: Site123\
Evidence: Found matches for s123-js, site123\.com\
Confidence Level: Medium\
Google Tag Manager not Detected \
Meta Pixel Detected: No\
No Meta Pixel signals found.